In [ ]:
from dotenv import load_dotenv, find_dotenv

assert load_dotenv(find_dotenv(usecwd=False)), "The .env file was not loaded."

import numpy as np
import pandas as pd
import torch
from drn import *

from generate_synthetic_dataset import generate_synthetic_gamma
from hyperparameter_tuning_objectives import (
    objective_cann,
    objective_ddr,
    objective_drn,
    objective_mdn,
)

from analysis_utils import (
    process_data_with_std,
    generate_latex_table_more_runs,
    plot_metrics_grid,
    get_nll_crps_rmse_ql
)

torch.set_num_threads(1)

In [ ]:
# Create a shared large test set (without standardising - it will need to be adjusted for each model later).
x_test_raw_shared, y_test_raw_shared, _, _ = generate_synthetic_gamma(10_000, seed=131313)
x_test_raw_shared = pd.DataFrame(x_test_raw_shared, columns=["X_1", "X_2"])

X_test_raw_shared = torch.tensor(x_test_raw_shared.values)
Y_test_raw_shared = torch.tensor(y_test_raw_shared.values)

# Table D.6

In [ ]:
glm_gamma_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}

glm_gamma_null_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_null_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_null_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_null_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_null_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_null_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_null_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_null_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}

glm_gamma_empty_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_empty_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_empty_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_empty_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_empty_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_empty_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_empty_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_ig_empty_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}

In [ ]:
distribution = "gamma"
for size, seed_num in zip(
    [1000, 3000, 6000] * 20, [i for i in range(20) for _ in range(3)]
):
    if size == 1000:
        proportion = 0.2
        hidden_size = 128
        dropout_rate = 0.5

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.05

    elif size == 3000:
        proportion = 0.1
        hidden_size = 256
        dropout_rate = 0.4

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.05

    elif size == 6000:
        proportion = 0.1
        hidden_size = 512
        dropout_rate = 0.3

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.05

    else:
        proportion = 0.05
        hidden_size = 512
        dropout_rate = 0.2

        num_hidden_layers = 3
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.05 / 5

    print(f"Size: {size}; Seed: {seed_num}")
    print(
        "-----------------------------------------------------------------------------------"
    )

    features, target, means, dispersion = generate_synthetic_gamma(size, seed=seed_num)

    (
        x_train,
        x_val,
        x_test,
        y_train,
        y_val,
        y_test,
        x_train_raw,
        x_val_raw,
        x_test_raw,
        num_features,
        cat_features,
        all_categories,
        ct,
    ) = split_and_preprocess(
        features, target, ["X_1", "X_2"], [], seed=42, num_standard=True
    )

    X_train = torch.Tensor(x_train.values)
    Y_train = torch.Tensor(y_train.values)
    X_val = torch.Tensor(x_val.values)
    Y_val = torch.Tensor(y_val.values)
    # X_test = torch.Tensor(x_test.values)
    # Y_test = torch.Tensor(y_test.values)

    train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
    val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

    glm_gamma = GLM(X_train.shape[1], distribution="gamma")
    glm_ig = GLM(X_train.shape[1], distribution="inversegaussian")
    if True:
        glm_gamma = glm_gamma.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = glm_ig.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian"
        )
        glm_ig.eval()

        glm_gamma_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="gamma", null_model=True
        )
        glm_ig_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian", null_model=True
        )

        glm_gamma_empty = GLM(p=2, distribution="gamma", default=True)
        glm_ig_empty = GLM(p=2, distribution="inversegaussian", default=True)

    nll_test_dict, crps_test_dict, rmse_test_dict, ql_90_test_dict = (
        get_nll_crps_rmse_ql(
            models=[
                glm_gamma_null,
                glm_gamma_empty,
                glm_ig_null,
                glm_ig_empty,
                glm_gamma,
                glm_ig,
            ],
            names=[
                "GLM_GA_NULL",
                "GLM_GA_EMPTY",
                "GLM_IG_NULL",
                "GLM_IG_EMPTY",
                "GLM_GA",
                "GLM_IG",
            ],
            X_test_data=X_test_raw_shared,
            Y_test_data=Y_test_raw_shared,
            y_train=y_train,
        )
    )

    glm_gamma_null_dict_nll[f"{size}"].append(nll_test_dict["GLM_GA_NULL"].item())
    glm_gamma_null_dict_crps[f"{size}"].append(crps_test_dict["GLM_GA_NULL"].item())
    glm_gamma_null_dict_rmse[f"{size}"].append(rmse_test_dict["GLM_GA_NULL"].item())
    glm_gamma_null_dict_ql90[f"{size}"].append(ql_90_test_dict["GLM_GA_NULL"].item())

    glm_ig_null_dict_nll[f"{size}"].append(nll_test_dict["GLM_IG_NULL"].item())
    glm_ig_null_dict_crps[f"{size}"].append(crps_test_dict["GLM_IG_NULL"].item())
    glm_ig_null_dict_rmse[f"{size}"].append(rmse_test_dict["GLM_IG_NULL"].item())
    glm_ig_null_dict_ql90[f"{size}"].append(ql_90_test_dict["GLM_IG_NULL"].item())

    glm_gamma_empty_dict_nll[f"{size}"].append(nll_test_dict["GLM_GA_EMPTY"].item())
    glm_gamma_empty_dict_crps[f"{size}"].append(crps_test_dict["GLM_GA_EMPTY"].item())
    glm_gamma_empty_dict_rmse[f"{size}"].append(rmse_test_dict["GLM_GA_EMPTY"].item())
    glm_gamma_empty_dict_ql90[f"{size}"].append(ql_90_test_dict["GLM_GA_EMPTY"].item())

    glm_ig_empty_dict_nll[f"{size}"].append(nll_test_dict["GLM_IG_EMPTY"].item())
    glm_ig_empty_dict_crps[f"{size}"].append(crps_test_dict["GLM_IG_EMPTY"].item())
    glm_ig_empty_dict_rmse[f"{size}"].append(rmse_test_dict["GLM_IG_EMPTY"].item())
    glm_ig_empty_dict_ql90[f"{size}"].append(ql_90_test_dict["GLM_IG_EMPTY"].item())

    glm_gamma_dict_nll[f"{size}"].append(nll_test_dict["GLM_GA"].item())
    glm_gamma_dict_crps[f"{size}"].append(crps_test_dict["GLM_GA"].item())
    glm_gamma_dict_rmse[f"{size}"].append(rmse_test_dict["GLM_GA"].item())
    glm_gamma_dict_ql90[f"{size}"].append(ql_90_test_dict["GLM_GA"].item())

    glm_ig_dict_nll[f"{size}"].append(nll_test_dict["GLM_IG"].item())
    glm_ig_dict_crps[f"{size}"].append(crps_test_dict["GLM_IG"].item())
    glm_ig_dict_rmse[f"{size}"].append(rmse_test_dict["GLM_IG"].item())
    glm_ig_dict_ql90[f"{size}"].append(ql_90_test_dict["GLM_IG"].item())

In [ ]:
data_dicts = {
    "NLL": {
        "GLM_GA": glm_gamma_dict_nll,
        "GLM_IG": glm_ig_dict_nll,
        "GLM_GA_NULL": glm_gamma_null_dict_nll,
        "GLM_IG_NULL": glm_ig_null_dict_nll,
        "GLM_GA_EMPTY": glm_gamma_empty_dict_nll,
        "GLM_IG_EMPTY": glm_ig_empty_dict_nll,
    },
    "CRPS": {
        "GLM_GA": glm_gamma_dict_crps,
        "GLM_IG": glm_ig_dict_crps,
        "GLM_GA_NULL": glm_gamma_null_dict_crps,
        "GLM_IG_NULL": glm_ig_null_dict_crps,
        "GLM_GA_EMPTY": glm_gamma_empty_dict_crps,
        "GLM_IG_EMPTY": glm_ig_empty_dict_crps,
    },
    "RMSE": {
        "GLM_GA": glm_gamma_dict_rmse,
        "GLM_IG": glm_ig_dict_rmse,
        "GLM_GA_NULL": glm_gamma_null_dict_rmse,
        "GLM_IG_NULL": glm_ig_null_dict_rmse,
        "GLM_GA_EMPTY": glm_gamma_empty_dict_rmse,
        "GLM_IG_EMPTY": glm_ig_empty_dict_rmse,
    },
    "QL90": {
        "GLM_GA": glm_gamma_dict_ql90,
        "GLM_IG": glm_ig_dict_ql90,
        "GLM_GA_NULL": glm_gamma_null_dict_ql90,
        "GLM_IG_NULL": glm_ig_null_dict_ql90,
        "GLM_GA_EMPTY": glm_gamma_empty_dict_ql90,
        "GLM_IG_EMPTY": glm_ig_empty_dict_ql90,
    },
}

metrics = ["NLL", "CRPS", "RMSE", "QL90"]
models = [
    "GLM_GA",
    "GLM_IG",
    "GLM_GA_NULL",
    "GLM_IG_NULL",
    "GLM_GA_EMPTY",
    "GLM_IG_EMPTY",
]

plot_metrics_grid(
    data_dicts, metrics, models, process_data_with_std, keys=[1000, 3000, 6000]
)
generate_latex_table_more_runs(
    data_dicts, metrics, models, keys=["1000", "3000", "6000"]
)

# Top Panel of Table D.7

In [ ]:
drn_gamma_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_ig_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_ig_kl_small_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
mdn_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}

drn_gamma_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_ig_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_ig_kl_small_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
mdn_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}

drn_gamma_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_ig_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_ig_kl_small_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
mdn_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}

drn_gamma_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_ig_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_ig_kl_small_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
mdn_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}

In [ ]:
distribution = "gamma"

for size, seed_num in zip(
    [1000, 3000, 6000] * 20, [i for i in range(20) for _ in range(3)]
):
    if size == 1000:
        proportion = 0.2
        hidden_size = 128
        dropout_rate = 0.5

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.05

    elif size == 3000:
        proportion = 0.1
        hidden_size = 256
        dropout_rate = 0.4

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.05

    elif size == 6000:
        proportion = 0.1
        hidden_size = 512
        dropout_rate = 0.3

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.05

    else:
        proportion = 0.05
        hidden_size = 512
        dropout_rate = 0.2

        num_hidden_layers = 3
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.05 / 5

    print(f"Size: {size}; Seed: {seed_num}")
    print(
        "-----------------------------------------------------------------------------------"
    )

    features, target, means, dispersion = generate_synthetic_gamma(size, seed=seed_num)

    (
        x_train,
        x_val,
        x_test,
        y_train,
        y_val,
        y_test,
        x_train_raw,
        x_val_raw,
        x_test_raw,
        num_features,
        cat_features,
        all_categories,
        ct,
    ) = split_and_preprocess(
        features, target, ["X_1", "X_2"], [], seed=42, num_standard=True
    )

    X_train = torch.Tensor(x_train.values)
    Y_train = torch.Tensor(y_train.values)
    X_val = torch.Tensor(x_val.values)
    Y_val = torch.Tensor(y_val.values)
    # X_test = torch.Tensor(x_test.values)
    # Y_test = torch.Tensor(y_test.values)

    train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
    val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

    glm_gamma = GLM(X_train.shape[1], distribution="gamma")
    glm_ig = GLM(X_train.shape[1], distribution="inversegaussian")
    if True:
        glm_gamma = glm_gamma.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = glm_ig.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian"
        )
        glm_ig.eval()

    cutpoints_DRN = drn_cutpoints(
        c_0=(
            np.min(Y_train.detach().numpy()) * 1.1
            if np.min(Y_train.detach().numpy()) < 0
            else 0.0
        ),
        c_K=20,
        p=proportion,
        y=Y_train.detach().numpy(),
        min_obs=3,
    )

    print(len(cutpoints_DRN))

    torch.manual_seed(23)
    drn_gamma = DRN(
        num_features=X_train.shape[1],
        cutpoints=cutpoints_DRN,
        glm=glm_gamma,
        hidden_size=hidden_size,
        num_hidden_layers=num_hidden_layers,
        baseline_start=False,
        dropout_rate=dropout_rate,
    )
    if True:
        torch.manual_seed(23)
        train(
            model=drn_gamma,
            criterion=lambda pred, y: drn_loss(
                pred,
                y,
                kl_alpha=kl_alpha,
                dv_alpha=0.0,
                kind="jbce",
                kl_direction="forwards",
            ),
            # criterion_val=lambda pred, y: drn_loss(pred, y, kl_alpha = 0.0, kind = 'jbce'),
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=50,
        )
        drn_gamma.eval()

    torch.manual_seed(23)
    drn_ig = DRN(
        num_features=X_train.shape[1],
        cutpoints=cutpoints_DRN,
        glm=glm_ig,
        hidden_size=hidden_size,
        num_hidden_layers=num_hidden_layers,
        baseline_start=False,
        dropout_rate=dropout_rate,
    )

    if True:
        torch.manual_seed(23)
        train(
            model=drn_ig,
            criterion=lambda pred, y: drn_loss(
                pred,
                y,
                kl_alpha=kl_alpha,
                dv_alpha=0.0,
                kind="jbce",
                kl_direction="forwards",
            ),
            # criterion_val=lambda pred, y: drn_loss(pred, y, kl_alpha = 0.0, kind = 'jbce'),
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_ig.eval()

    torch.manual_seed(23)
    drn_ig_small_kl = DRN(
        num_features=X_train.shape[1],
        cutpoints=cutpoints_DRN,
        glm=glm_ig,
        hidden_size=hidden_size,
        num_hidden_layers=num_hidden_layers,
        baseline_start=False,
        dropout_rate=dropout_rate,
    )

    if True:
        torch.manual_seed(23)
        train(
            model=drn_ig_small_kl,
            criterion=lambda pred, y: drn_loss(
                pred,
                y,
                kl_alpha=kl_alpha,
                dv_alpha=0.0,
                kind="jbce",
                kl_direction="forwards",
            ),
            # criterion_val=lambda pred, y: drn_loss(pred, y, kl_alpha = 0.0, kind = 'jbce'),
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_ig_small_kl.eval()

    # torch.manual_seed(23)
    # cann_gamma = CANN(glm_gamma,
    #                 num_hidden_layers=num_hidden_layers,
    #                     hidden_size=hidden_size,
    #                     dropout_rate =dropout_rate)

    # if True:
    #     torch.manual_seed(23)
    #     train(
    #         cann_gamma,
    #         gaussian_deviance_loss if distribution == "gaussian" else gamma_deviance_loss,
    #         train_dataset,
    #         val_dataset,
    #         epochs=5000,
    #         lr=lr,
    #         patience=patience,
    #         batch_size=batch_size,
    #         log_interval=100,
    #     )
    #     cann_gamma.update_dispersion(X_train, Y_train)
    #     cann_gamma.eval()

    torch.manual_seed(23)
    ddr = DDR(
        x_train.shape[1],
        cutpoints_DRN,
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
    )  # , glm = glm)
    if True:
        torch.manual_seed(23)
        train(
            ddr,
            ddr_loss,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            lr=lr,
            batch_size=batch_size,
            log_interval=100,
            patience=patience,
            epochs=5000,
        )
        ddr.eval()

    torch.manual_seed(23)
    mdn = MDN(
        X_train.shape[1],
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
        num_components=5,
        distribution=distribution,
    )

    if True:
        torch.manual_seed(23)
        train(
            mdn,
            gaussian_mdn_loss if distribution == "gaussian" else gamma_mdn_loss,
            train_dataset,
            val_dataset,
            lr=lr,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            log_interval=100,
        )
        mdn.eval()

    nll_test_dict, crps_test_dict, rmse_test_dict, ql_90_test_dict = (
        get_nll_crps_rmse_ql(
            models=[drn_ig, drn_ig_small_kl, ddr, mdn, drn_gamma],
            names=["DRN_IG", "DRN_IG_KL_SMALL", "DDR", "MDN", "DRN_GA"],
            X_test_data=X_test_raw_shared,
            Y_test_data=Y_test_raw_shared,
            y_train=y_train,
        )
    )

    drn_ig_dict_nll[f"{size}"].append(nll_test_dict["DRN_IG"].item())
    drn_ig_dict_crps[f"{size}"].append(crps_test_dict["DRN_IG"].item())
    drn_ig_dict_rmse[f"{size}"].append(rmse_test_dict["DRN_IG"].item())
    drn_ig_dict_ql90[f"{size}"].append(ql_90_test_dict["DRN_IG"].item())

    drn_ig_kl_small_dict_nll[f"{size}"].append(nll_test_dict["DRN_IG_KL_SMALL"].item())
    drn_ig_kl_small_dict_crps[f"{size}"].append(
        crps_test_dict["DRN_IG_KL_SMALL"].item()
    )
    drn_ig_kl_small_dict_rmse[f"{size}"].append(
        rmse_test_dict["DRN_IG_KL_SMALL"].item()
    )
    drn_ig_kl_small_dict_ql90[f"{size}"].append(
        ql_90_test_dict["DRN_IG_KL_SMALL"].item()
    )

    ddr_dict_nll[f"{size}"].append(nll_test_dict["DDR"].item())
    ddr_dict_crps[f"{size}"].append(crps_test_dict["DDR"].item())
    ddr_dict_rmse[f"{size}"].append(rmse_test_dict["DDR"].item())
    ddr_dict_ql90[f"{size}"].append(ql_90_test_dict["DDR"].item())

    mdn_dict_nll[f"{size}"].append(nll_test_dict["MDN"].item())
    mdn_dict_crps[f"{size}"].append(crps_test_dict["MDN"].item())
    mdn_dict_rmse[f"{size}"].append(rmse_test_dict["MDN"].item())
    mdn_dict_ql90[f"{size}"].append(ql_90_test_dict["MDN"].item())

    drn_gamma_dict_nll[f"{size}"].append(nll_test_dict["DRN_GA"].item())
    drn_gamma_dict_crps[f"{size}"].append(crps_test_dict["DRN_GA"].item())
    drn_gamma_dict_rmse[f"{size}"].append(rmse_test_dict["DRN_GA"].item())
    drn_gamma_dict_ql90[f"{size}"].append(ql_90_test_dict["DRN_GA"].item())

    # cann_dict_nll[f'{size}'].append(nll_test_dict['CANN'].item())
    # cann_dict_crps[f'{size}'].append(crps_test_dict['CANN'].item())
    # cann_dict_rmse[f'{size}'].append(rmse_test_dict['CANN'].item())
    # cann_dict_ql90[f'{size}'].append(ql_90_test_dict['DDR'].item())

    # glm_gamma_dict_nll[f'{size}'].append(nll_test_dict['GLM_GA'].item())
    # glm_gamma_dict_crps[f'{size}'].append(crps_test_dict['GLM_GA'].item())
    # glm_gamma_dict_rmse[f'{size}'].append(rmse_test_dict['GLM_GA'].item())
    # glm_gamma_dict_ql90[f'{size}'].append(ql_90_test_dict['GLM_GA'].item())

    # glm_ig_dict_nll[f'{size}'].append(nll_test_dict['GLM_IG'].item())
    # glm_ig_dict_crps[f'{size}'].append(crps_test_dict['GLM_IG'].item())
    # glm_ig_dict_rmse[f'{size}'].append(rmse_test_dict['GLM_IG'].item())
    # glm_ig_dict_ql90[f'{size}'].append(ql_90_test_dict['GLM_IG'].item())

    # print('-----------------------------------------------------------------------------------')

In [ ]:
# Define evaluation dictionaries for automation
nll_test_dict = {
    "DRN_IG": drn_ig_dict_nll,
    "DRN_IG_KL_SMALL": drn_ig_kl_small_dict_nll,
    "DDR": ddr_dict_nll,
    "MDN": mdn_dict_nll,
    "DRN_GA": drn_gamma_dict_nll,
}

crps_test_dict = {
    "DRN_IG": drn_ig_dict_crps,
    "DRN_IG_KL_SMALL": drn_ig_kl_small_dict_crps,
    "DDR": ddr_dict_crps,
    "MDN": mdn_dict_crps,
    "DRN_GA": drn_gamma_dict_crps,
}

rmse_test_dict = {
    "DRN_IG": drn_ig_dict_rmse,
    "DRN_IG_KL_SMALL": drn_ig_kl_small_dict_rmse,
    "DDR": ddr_dict_rmse,
    "MDN": mdn_dict_rmse,
    "DRN_GA": drn_gamma_dict_rmse,
}

ql_90_test_dict = {
    "DRN_IG": drn_ig_dict_ql90,
    "DRN_IG_KL_SMALL": drn_ig_kl_small_dict_ql90,
    "DDR": ddr_dict_ql90,
    "MDN": mdn_dict_ql90,
    "DRN_GA": drn_gamma_dict_ql90,
}

data_dicts = {
    "NLL": nll_test_dict,
    "CRPS": crps_test_dict,
    "RMSE": rmse_test_dict,
    "QL90": ql_90_test_dict,
}


metrics = ["NLL", "CRPS", "RMSE", "QL90"]
models = ["DRN_IG", "DRN_IG_KL_SMALL", "DDR", "MDN", "DRN_GA"]

plot_metrics_grid(
    data_dicts, metrics, models, process_data_with_std, keys=[1000, 3000, 6000]
)
generate_latex_table_more_runs(
    data_dicts, metrics, models, keys=["1000", "3000", "6000"]
)

# Bottom Panel of Table D.7

In [ ]:
drn_gamma_backward_null_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_gamma_backward_empty_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_null_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_empty_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
# drn_ig_null_nll = {'1000': [], '3000': [], '6000' : [], '20000' : []}
# drn_ig_empty_nll = {'1000': [], '3000': [], '6000' : [], '20000' : []}

drn_gamma_backward_null_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_gamma_backward_empty_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_null_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_empty_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
# drn_ig_null_crps = {'1000': [], '3000': [], '6000' : [], '20000' : []}
# drn_ig_empty_crps = {'1000': [], '3000': [], '6000' : [], '20000' : []}

drn_gamma_backward_null_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_gamma_backward_empty_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_null_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_empty_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
# drn_ig_null_rmse = {'1000': [], '3000': [], '6000' : [], '20000' : []}
# drn_ig_empty_rmse = {'1000': [], '3000': [], '6000' : [], '20000' : []}

drn_gamma_backward_null_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
drn_gamma_backward_empty_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_null_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_empty_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
# drn_ig_null_ql90 = {'1000': [], '3000': [], '6000' : [], '20000' : []}
# drn_ig_empty_ql90 = {'1000': [], '3000': [], '6000' : [], '20000' : []}

In [ ]:
distribution = "gamma"
for size, seed_num in zip(
    [1000, 3000, 6000] * 20, [i for i in range(20) for _ in range(3)]
):
    # for size, seed_num in zip([1000]*1, [i for i in range(100, 101) for _ in range(1)]):
    if size == 1000:
        proportion = 0.2 * 2
        hidden_size = 128
        dropout_rate = 0.5

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.01

    elif size == 3000:
        proportion = 0.1
        hidden_size = 256
        dropout_rate = 0.4

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.01

    elif size == 6000:
        proportion = 0.1
        hidden_size = 512
        dropout_rate = 0.3

        num_hidden_layers = 2
        lr = 1e-3 / 4
        batch_size = 128
        patience = 100
        kl_alpha = 0.01

    # else:
    #     proportion = 0.05
    #     hidden_size = 512
    #     dropout_rate = 0.2

    #     num_hidden_layers = 3
    #     lr = 1e-3/4
    #     batch_size = 128
    #     patience = 100
    #     kl_alpha = 0.05/2

    print(f"Size: {size}; Seed: {seed_num}")
    print(
        "-----------------------------------------------------------------------------------"
    )

    features, target, means, dispersion = generate_synthetic_gamma(size, seed=seed_num)

    (
        x_train,
        x_val,
        x_test,
        y_train,
        y_val,
        y_test,
        x_train_raw,
        x_val_raw,
        x_test_raw,
        num_features,
        cat_features,
        all_categories,
        ct,
    ) = split_and_preprocess(
        features, target, ["X_1", "X_2"], [], seed=42, num_standard=True
    )

    X_train = torch.Tensor(x_train.values)
    Y_train = torch.Tensor(y_train.values)
    X_val = torch.Tensor(x_val.values)
    Y_val = torch.Tensor(y_val.values)
    # X_test = torch.Tensor(x_test.values)
    # Y_test = torch.Tensor(y_test.values)

    train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
    val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

    glm_gamma = GLM(X_train.shape[1], distribution="gamma")
    glm_ig = GLM(X_train.shape[1], distribution="inversegaussian")
    if True:
        glm_gamma = glm_gamma.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = glm_ig.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian"
        )
        glm_ig.eval()

        glm_gamma_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="gamma", null_model=True
        )
        glm_ig_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian", null_model=True
        )

        glm_gamma_empty = GLM(p=2, distribution="gamma", default=True)
        glm_ig_empty = GLM(p=2, distribution="inversegaussian", default=True)

        # glm_gamma_wrong= GLM(p=2, distribution="gamma", default=True)
        # glm_gamma_wrong.linear.weight = nn.Parameter(torch.tensor([[0.0, 0.0]]), requires_grad=True)
        # glm_gamma_wrong.linear.bias = nn.Parameter(torch.tensor([-2.0]), requires_grad=False)
        # glm_gamma_wrong.dispersion = nn.Parameter(torch.tensor([5.0]), requires_grad=False)

    cutpoints_DRN = drn_cutpoints(
        c_0=(
            np.min(Y_train.detach().numpy()) * 1.1
            if np.min(Y_train.detach().numpy()) < 0
            else 0.0
        ),
        c_K=20,  # np.max(Y_train.detach().numpy()) * 1.1,
        p=proportion,
        y=Y_train.detach().numpy(),
        min_obs=3,
    )
    print(len(cutpoints_DRN))

    torch.manual_seed(23)
    drn_gamma_null = DRN(
        num_features=X_train.shape[1],
        cutpoints=cutpoints_DRN,
        glm=glm_gamma_null,
        hidden_size=hidden_size,
        num_hidden_layers=num_hidden_layers,
        baseline_start=False,
        dropout_rate=dropout_rate,
    )
    if True:
        torch.manual_seed(23)
        train(
            model=drn_gamma_null,
            criterion=lambda pred, y: drn_loss(
                pred,
                y,
                kl_alpha=kl_alpha,
                dv_alpha=0.0,
                kind="jbce",
                kl_direction="backward",
            ),
            # criterion_val=lambda pred, y: drn_loss(pred, y, kl_alpha = 0.0, kind = 'jbce'),
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_gamma_null.eval()

    torch.manual_seed(23)
    drn_gamma_empty = DRN(
        num_features=X_train.shape[1],
        cutpoints=cutpoints_DRN,
        glm=glm_gamma_empty,
        hidden_size=hidden_size,
        num_hidden_layers=num_hidden_layers,
        baseline_start=False,
        dropout_rate=dropout_rate,
    )
    if True:
        torch.manual_seed(23)
        train(
            model=drn_gamma_empty,
            criterion=lambda pred, y: drn_loss(
                pred,
                y,
                kl_alpha=kl_alpha,
                dv_alpha=0.0,
                kind="jbce",
                kl_direction="backward",
            ),
            # criterion_val=lambda pred, y: drn_loss(pred, y, kl_alpha = 0.0, kind = 'jbce'),
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_gamma_empty.eval()

    torch.manual_seed(23)
    cann_gamma_null = CANN(
        glm_gamma_null,
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
    )

    if True:
        torch.manual_seed(23)
        train(
            cann_gamma_null,
            (
                gaussian_deviance_loss
                if distribution == "gaussian"
                else gamma_deviance_loss
            ),
            train_dataset,
            val_dataset,
            epochs=5000,
            lr=lr,
            patience=patience,
            batch_size=batch_size,
            log_interval=100,
        )
        cann_gamma_null.update_dispersion(X_train, Y_train)
        cann_gamma_null.eval()

    torch.manual_seed(23)
    cann_gamma_empty = CANN(
        glm_gamma_empty,
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
    )

    if True:
        torch.manual_seed(23)
        train(
            cann_gamma_empty,
            (
                gaussian_deviance_loss
                if distribution == "gaussian"
                else gamma_deviance_loss
            ),
            train_dataset,
            val_dataset,
            epochs=5000,
            lr=lr,
            patience=patience,
            batch_size=batch_size,
            log_interval=100,
        )
        cann_gamma_empty.update_dispersion(X_train, Y_train)
        cann_gamma_empty.eval()

    # torch.manual_seed(23)
    # drn_ig_null = DRN(num_features = X_train.shape[1], cutpoints = cutpoints_DRN, glm = glm_ig_null,\
    #                 hidden_size=hidden_size, num_hidden_layers=num_hidden_layers,
    #                   baseline_start = True,  dropout_rate = dropout_rate)

    # if True:
    #     torch.manual_seed(23)
    #     train(
    #                 model=drn_ig_null,
    #                 criterion=lambda pred, y: drn_loss(pred, y, kl_alpha = kl_alpha, dv_alpha=0.0,
    #                                                         kind = 'jbce', kl_direction='forwards'),
    #                 criterion_val=lambda pred, y: drn_loss(pred, y, kl_alpha = 0.0, kind = 'jbce'),
    #                 train_dataset=train_dataset,
    #                 val_dataset=val_dataset,
    #                 batch_size=batch_size,
    #                 epochs=5000,
    #                 patience=patience,
    #                 lr=lr,
    #                 print_details=True,
    #                 log_interval=100,
    #     )
    #     drn_ig_null.eval()

    # torch.manual_seed(23)
    # drn_ig_empty = DRN(num_features = X_train.shape[1], cutpoints = cutpoints_DRN, glm = glm_ig_empty,\
    #                 hidden_size=hidden_size, num_hidden_layers=num_hidden_layers,
    #                   baseline_start = True,  dropout_rate = dropout_rate)

    # if True:
    #     torch.manual_seed(23)
    #     train(
    #                 model=drn_ig_empty,
    #                 criterion=lambda pred, y: drn_loss(pred, y, kl_alpha = kl_alpha, dv_alpha=0.0,
    #                                                         kind = 'jbce', kl_direction='forwards'),
    #                 criterion_val=lambda pred, y: drn_loss(pred, y, kl_alpha = 0.0, kind = 'jbce'),
    #                 train_dataset=train_dataset,
    #                 val_dataset=val_dataset,
    #                 batch_size=batch_size,
    #                 epochs=5000,
    #                 patience=patience,
    #                 lr=lr,
    #                 print_details=True,
    #                 log_interval=100,
    #     )
    #     drn_ig_empty.eval()

    nll_test_dict, crps_test_dict, rmse_test_dict, ql_90_test_dict = (
        get_nll_crps_rmse_ql(
            models=[drn_gamma_empty, drn_gamma_null, cann_gamma_empty, cann_gamma_null],
            names=["DRN_GA_EMPTY", "DRN_GA_NULL", "CANN_GA_EMPTY", "CANN_GA_NULL"],
            X_test_data=X_test_raw_shared,
            Y_test_data=Y_test_raw_shared,
            y_train=y_train,
        )
    )

    # drn_gamma_null_nll[f'{size}'].append(nll_test_dict['DRN_GA_NULL'].item())
    # drn_gamma_null_crps[f'{size}'].append(crps_test_dict['DRN_GA_NULL'].item())
    # drn_gamma_null_rmse[f'{size}'].append(rmse_test_dict['DRN_GA_NULL'].item())
    # drn_gamma_null_ql90[f'{size}'].append(ql_90_test_dict['DRN_GA_NULL'].item())

    # drn_gamma_empty_nll[f'{size}'].append(nll_test_dict['DRN_GA_EMPTY'].item())
    # drn_gamma_empty_crps[f'{size}'].append(crps_test_dict['DRN_GA_EMPTY'].item())
    # drn_gamma_empty_rmse[f'{size}'].append(rmse_test_dict['DRN_GA_EMPTY'].item())
    # drn_gamma_empty_ql90[f'{size}'].append(ql_90_test_dict['DRN_GA_EMPTY'].item())

    drn_gamma_backward_null_nll[f"{size}"].append(nll_test_dict["DRN_GA_NULL"].item())
    drn_gamma_backward_null_crps[f"{size}"].append(crps_test_dict["DRN_GA_NULL"].item())
    drn_gamma_backward_null_rmse[f"{size}"].append(rmse_test_dict["DRN_GA_NULL"].item())
    drn_gamma_backward_null_ql90[f"{size}"].append(
        ql_90_test_dict["DRN_GA_NULL"].item()
    )

    drn_gamma_backward_empty_nll[f"{size}"].append(nll_test_dict["DRN_GA_EMPTY"].item())
    drn_gamma_backward_empty_crps[f"{size}"].append(
        crps_test_dict["DRN_GA_EMPTY"].item()
    )
    drn_gamma_backward_empty_rmse[f"{size}"].append(
        rmse_test_dict["DRN_GA_EMPTY"].item()
    )
    drn_gamma_backward_empty_ql90[f"{size}"].append(
        ql_90_test_dict["DRN_GA_EMPTY"].item()
    )

    cann_gamma_null_nll[f"{size}"].append(nll_test_dict["CANN_GA_NULL"].item())
    cann_gamma_null_crps[f"{size}"].append(crps_test_dict["CANN_GA_NULL"].item())
    cann_gamma_null_rmse[f"{size}"].append(rmse_test_dict["CANN_GA_NULL"].item())
    cann_gamma_null_ql90[f"{size}"].append(ql_90_test_dict["CANN_GA_NULL"].item())

    cann_gamma_empty_nll[f"{size}"].append(nll_test_dict["CANN_GA_EMPTY"].item())
    cann_gamma_empty_crps[f"{size}"].append(crps_test_dict["CANN_GA_EMPTY"].item())
    cann_gamma_empty_rmse[f"{size}"].append(rmse_test_dict["CANN_GA_EMPTY"].item())
    cann_gamma_empty_ql90[f"{size}"].append(ql_90_test_dict["CANN_GA_EMPTY"].item())

In [ ]:
# Simulated data dictionaries (replace these with actual values)
data_dicts = {
    "NLL": {
        "DRN_GA_NULL (REVERSE KL)": drn_gamma_backward_null_nll,
        "DRN_GA_BIASED (REVERSE KL)": drn_gamma_backward_empty_nll,
        "CANN_GA_NULL": cann_gamma_null_nll,
        "CANN_GA_BIASED": cann_gamma_empty_nll,
    },
    "CRPS": {
        "DRN_GA_NULL (REVERSE KL)": drn_gamma_backward_null_crps,
        "DRN_GA_BIASED (REVERSE KL)": drn_gamma_backward_empty_crps,
        "CANN_GA_NULL": cann_gamma_null_crps,
        "CANN_GA_BIASED": cann_gamma_empty_crps,
    },
    "RMSE": {
        "DRN_GA_NULL (REVERSE KL)": drn_gamma_backward_null_rmse,
        "DRN_GA_BIASED (REVERSE KL)": drn_gamma_backward_empty_rmse,
        "CANN_GA_NULL": cann_gamma_null_rmse,
        "CANN_GA_BIASED": cann_gamma_empty_rmse,
    },
    "QL90": {
        "DRN_GA_NULL (REVERSE KL)": drn_gamma_backward_null_ql90,
        "DRN_GA_BIASED (REVERSE KL)": drn_gamma_backward_empty_ql90,
        "CANN_GA_NULL": cann_gamma_null_ql90,
        "CANN_GA_BIASED": cann_gamma_empty_ql90,
    },
}

metrics = ["NLL", "CRPS", "RMSE", "QL90"]
models = [
    "CANN_GA_BIASED",
    "CANN_GA_NULL",
    "DRN_GA_BIASED (REVERSE KL)",
    "DRN_GA_NULL (REVERSE KL)",
]

plot_metrics_grid(
    data_dicts, metrics, models, process_data_with_std, keys=[1000, 3000, 6000]
)
generate_latex_table_more_runs(
    data_dicts, metrics, models, keys=["1000", "3000", "6000"]
)